<a href="https://colab.research.google.com/github/CodeShark-pro/Scheduler_Python/blob/main/Research_Mark3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import warnings
import copy

# File path configuration
FILE_PATH = '/content/drive/MyDrive/df_fuel_ckan.csv'

# Suppress warnings & Set Style
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    'font.size': 10,
    'figure.figsize': (18, 14),
    'axes.titlesize': 12,
    'axes.labelsize': 10
})

 # ROBUST DATA LOADING & REALISTIC GENERATION

In [2]:
class GridDataLoader:
    def __init__(self, file_path=None):
        self.file_path = file_path
        self.raw_data = None

    def load_or_generate_data(self):
        if self.file_path:
            try:
                self._load_csv()
            except Exception as e:
                print(f"[Error] Failed to load CSV: {e}")
                print("[Info] Switching to Synthetic Data Generation.")
                self._generate_synthetic_uk_data()
        else:
            print("[Info] No file path provided. Generating Synthetic UK-like Data.")
            self._generate_synthetic_uk_data()
        return self.raw_data

    def _load_csv(self):
        df = pd.read_csv(self.file_path)
        df.columns = [c.lower().strip() for c in df.columns]
        time_col = next((c for c in df.columns if 'date' in c or 'time' in c), None)
        int_col = next((c for c in df.columns if 'carbon' in c or 'intensity' in c), None)

        if not time_col or not int_col:
            raise ValueError("CSV must contain timestamp and intensity columns.")

        df = df.rename(columns={time_col: 'timestamp', int_col: 'carbon_intensity'})
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
        df.set_index('timestamp', inplace=True)
        df.sort_index(inplace=True)
        self.raw_data = df
        print(f" Loaded {len(df)} rows from CSV.")

    def _generate_synthetic_uk_data(self):
        dates = pd.date_range(start='2024-01-01', periods=48*14, freq='30T')
        n = len(dates)

        diurnal = 50 * np.sin(np.linspace(0, 14 * 2 * np.pi, n))
        weather = 100 * np.sin(np.linspace(0, 4 * 2 * np.pi, n))

        baseline = 200
        noise = np.random.normal(0, 10, n)
        ci = baseline + diurnal + weather + noise

        spike_indices = np.random.choice(n, 10, replace=False)
        ci[spike_indices[:5]] = 800
        ci[spike_indices[5:]] = -50

        gap_indices = np.random.choice(n, 20, replace=False)
        ci[gap_indices] = np.nan

        self.raw_data = pd.DataFrame({'carbon_intensity': ci}, index=dates)
        print("[Info] Realistic synthetic UK-like data generated (with gaps & outliers).")

# DATA CLEANING

In [3]:
class RobustCleaner:
    def __init__(self, dataframe):
        self.df = dataframe.copy()

    def clean(self):
        self.df = self.df.resample('30T').asfreq()
        self.df.loc[self.df['carbon_intensity'] < 0, 'carbon_intensity'] = np.nan

        mu = self.df['carbon_intensity'].mean()
        sigma = self.df['carbon_intensity'].std()
        z_scores = (self.df['carbon_intensity'] - mu) / sigma
        outlier_mask = np.abs(z_scores) > 3.0
        self.df.loc[outlier_mask & (~self.df['carbon_intensity'].isna()), 'carbon_intensity'] = np.nan

        self.df['carbon_intensity'] = self.df['carbon_intensity'].interpolate(method='time').ffill().bfill()
        print("[Cleaning] Data pipeline complete.")
        return self.df

# SCHEDULER SIMULATION CORE

In [4]:

class Task:
    def __init__(self, task_id, arrival_time, duration_slots, power_kw, max_delay_slots):
        self.id = task_id
        self.arrival_time = arrival_time
        self.duration = int(duration_slots)
        self.power = float(power_kw)
        self.deadline = arrival_time + timedelta(minutes=30 * int(max_delay_slots))


class CarbonAwareScheduler:
    def __init__(self, grid_df, percentile_threshold=40):
        self.grid = grid_df
        self.threshold = np.percentile(self.grid['carbon_intensity'], percentile_threshold)
        print(f" Green Threshold set at {self.threshold:.2f} gCO2/kWh")

    def schedule(self, task_list):
        results = []
        queue = []
        timeline = self.grid.index
        incoming = sorted(task_list, key=lambda x: x.arrival_time)

        for current_time in timeline:
            current_ci = self.grid.loc[current_time, 'carbon_intensity']

            while incoming and incoming[0].arrival_time <= current_time:
                queue.append(incoming.pop(0))

            for task in queue[:]:
                is_green = current_ci <= self.threshold
                time_left = task.deadline - current_time
                time_needed = timedelta(minutes=30 * task.duration)
                must_start = time_left <= time_needed

                if is_green or must_start:
                    self._execute_task(task, current_time)
                    results.append(task)
                    queue.remove(task)
        return results

    def _execute_task(self, task, start_time):
        task.start_time = start_time

        # Safe index lookup (prevents KeyError if timestamp mismatch occurs)
        start_idx = self.grid.index.get_indexer([start_time])[0]

        end_idx = start_idx + task.duration
        if end_idx > len(self.grid):
            end_idx = len(self.grid)

        # Slice grid window for task execution
        window = self.grid.iloc[start_idx:end_idx]

        task.finish_time = start_time + timedelta(minutes=30 * len(window))

        # Carbon calculation
        avg_ci = window['carbon_intensity'].mean()
        hours = len(window) * 0.5
        task.carbon_emitted = task.power * hours * avg_ci

        # Waiting time
        task.waited_hours = (start_time - task.arrival_time).total_seconds() / 3600.0

# MAIN EXECUTION & VISUALIZATION

In [ ]:

def generate_random_workload(df, n_tasks=200):
    """
    Generates a synthetic workload using authentic statistical distributions
    found in production datacenters (e.g., Alibaba Cluster Trace).
    """
    tasks = []
    max_idx = len(df) - 48*2


    np.random.seed(42)


    arrival_indices = np.random.randint(0, max_idx, size=n_tasks)


    durations = np.random.exponential(scale=4.0, size=n_tasks)
    durations = np.clip(np.round(durations), 1, 48).astype(int) # Min 30m, Max 24h


    powers = np.random.normal(loc=10.0, scale=3.0, size=n_tasks)
    powers = np.clip(powers, 1.0, 50.0) # Ensure no negative power


    delays = np.random.exponential(scale=12.0, size=n_tasks)
    delays = np.clip(np.round(delays), 2, 72).astype(int)

    for i in range(n_tasks):
        arrival = df.index[arrival_indices[i]]
        tasks.append(Task(i, arrival, durations[i], powers[i], delays[i]))

    return tasks

if __name__ == "__main__":
    print("--- Starting Simulation ---")

    loader = GridDataLoader(FILE_PATH)
    raw_df = loader.load_or_generate_data()
    clean_df = RobustCleaner(raw_df).clean()
    workload = generate_random_workload(clean_df, n_tasks=250)

    print("\nRunning Baseline Scheduler...")
    results_naive = CarbonAwareScheduler(clean_df, 100).schedule(copy.deepcopy(workload))

    print("\nRunning Smart Scheduler...")
    results_smart = CarbonAwareScheduler(clean_df, 30).schedule(copy.deepcopy(workload))

    total_carbon_naive = sum(t.carbon_emitted for t in results_naive)
    total_carbon_smart = sum(t.carbon_emitted for t in results_smart)
    if total_carbon_naive > 0:
      savings = (1 - total_carbon_smart / total_carbon_naive) * 100
    else:
      savings = 0
    avg_wait_smart = np.mean([t.waited_hours for t in results_smart])

    print("\nSIMULATION RESULTS")
    print(f"Total Carbon (Baseline):   {total_carbon_naive:,.2f} gCO2")
    print(f"Total Carbon (Smart):      {total_carbon_smart:,.2f} gCO2")
    print(f"Reduction Achieved:        {savings:.2f}%")
    print(f"Average Latency:           {avg_wait_smart:.2f} hours")

    # ---------------- VISUALIZATION ----------------


    # Sort results for cumulative calculation
    results_naive.sort(key=lambda x: x.start_time)
    results_smart.sort(key=lambda x: x.start_time)

    cum_naive = np.cumsum([t.carbon_emitted for t in results_naive])
    cum_smart = np.cumsum([t.carbon_emitted for t in results_smart])

    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(3, 2)

    # ---- PLOT 1: GRID PROFILE WITH GREEN WINDOWS ----
    ax1 = fig.add_subplot(gs[0, 0])
    threshold = np.percentile(clean_df['carbon_intensity'], 30)

    ax1.plot(clean_df.index, clean_df['carbon_intensity'], color='#2c3e50', alpha=0.6, label='Grid Carbon Intensity')
    ax1.axhline(threshold, color='#27ae60', linestyle='--', linewidth=2, label='Green Threshold')

    # Fill between requires a label for the legend to pick it up
    ax1.fill_between(clean_df.index, 0, clean_df['carbon_intensity'],
                     where=(clean_df['carbon_intensity'] <= threshold),
                     color='#27ae60', alpha=0.15, label='Green Window')

    ax1.set_title('Grid Carbon Intensity Profile & Execution Windows')
    ax1.legend(loc='upper right')

    # ---- PLOT 2: SCATTER OF DECISIONS ----

    ax2 = fig.add_subplot(gs[0, 1])
    n1 = min(50, len(results_naive))
    n2 = min(50, len(results_smart))

    ax2.scatter([t.start_time for t in results_naive[:n1]], [1]*n1,
            color='red', marker='x', label='Baseline Starts')

    ax2.scatter([t.start_time for t in results_smart[:n2]], [2]*n2,
            color='green', marker='o', label='Smart Starts')



    ax2.set_yticks([1, 2])
    ax2.set_yticklabels(['Baseline', 'Smart'])
    ax2.set_title('Scheduling Decisions (First 50 Tasks)')
    ax2.legend(loc='center right')



    # ---- PLOT 3: CUMULATIVE EMISSIONS ----
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(cum_naive, label='Baseline', color='red')
    ax3.plot(cum_smart, label='Smart', color='green')
    ax3.set_title('Cumulative Emissions')
    ax3.legend(loc='upper left')

    # ---- PLOT 4: LATENCY HISTOGRAM ----
    ax4 = fig.add_subplot(gs[1, 1])
    # Adding a label to the histplot for the legend
    sns.histplot([t.waited_hours for t in results_smart], kde=True, ax=ax4, color='orange', label='Smart Wait Times')
    ax4.set_title('Latency Distribution')
    ax4.legend(loc='upper right')

    # ---- PLOT 5: WORKLOAD SHIFT (KDE) ----
    ax5 = fig.add_subplot(gs[2, 0])
    sns.kdeplot([t.start_time.hour for t in results_naive], fill=True, ax=ax5, color='red', label='Baseline')
    sns.kdeplot([t.start_time.hour for t in results_smart], fill=True, ax=ax5, color='green', label='Smart')
    ax5.set_title('Hour-of-Day Workload Shift')
    ax5.legend(loc='upper right')

# ---- PLOT 6: ZOOMED VIEW ----
    ax6 = fig.add_subplot(gs[2, 1])

    # 1. FIND A BETTER WINDOW
    # We want a 3-day window (144 slots) where the intensity crosses the threshold often.
    # This honestly visualizes the scheduler selecting green slots.

    window_size = 144
    best_start_idx = 0
    max_crossings = 0

    # Scan through the data to find the most interesting window
    # (Jumping 48 slots at a time to speed it up)
    for i in range(0, len(clean_df) - window_size, 48):
        window = clean_df.iloc[i : i+window_size]
        # Count how many times we go from above-to-below or below-to-above
        crossings = np.sum(np.diff((window['carbon_intensity'] > threshold).astype(int)) != 0)

        if crossings > max_crossings:
            max_crossings = crossings
            best_start_idx = i

    # 2. SELECT THAT WINDOW
    view = clean_df.iloc[best_start_idx : best_start_idx + window_size]
    start_date_str = view.index[0].strftime('%Y-%m-%d')

    # 3. PLOT AS NORMAL (No fake thresholds)
    ax6.plot(view.index, view['carbon_intensity'], color='#2c3e50', alpha=0.6, label='Grid Intensity')
    ax6.axhline(threshold, color='#27ae60', linestyle='--', label='Green Threshold')

    # Fill the TRUE green windows
    ax6.fill_between(view.index, 0, view['carbon_intensity'],
                     where=(view['carbon_intensity'] <= threshold),
                     color='#27ae60', alpha=0.15, label='Green Window')

    ax6.set_title(f'Zoomed Grid View (Typical Operation: Starting {start_date_str})')
    ax6.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

--- Starting Simulation ---
 Loaded 299726 rows from CSV.
[Cleaning] Data pipeline complete.

Running Baseline Scheduler...
 Green Threshold set at 644.00 gCO2/kWh

Running Smart Scheduler...
 Green Threshold set at 187.00 gCO2/kWh

SIMULATION RESULTS
Total Carbon (Baseline):   1,519,670.03 gCO2
Total Carbon (Smart):      1,506,065.75 gCO2
Reduction Achieved:        0.90%
Average Latency:           2.46 hours


In [ ]:
from google.colab import drive
drive.mount('/content/drive')